## I. Các gói thư viện sử dụng

In [18]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [49]:
!pip install python-dotenv huggingface_hub datasets underthesea langdetect -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.5 MB/s eta 0:00:00


In [50]:
import os
import re
import numpy as np
import pandas as pd
from underthesea import word_tokenize
from langdetect import detect, detect_langs
from collections import Counter
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import login
from datasets import load_dataset

## II. Thiết lập môi trường làm việc
1. Kết nối google drive
2. Nạp biến môi trường từ .env
3. Kết nối Hugging Face

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [15]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [19]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

## III. Tải và trích xuất dữ liệu nhãn 0 từ ICCIES-2025-DetectAI/vietnamese_news_human_ai

In [80]:
dataset_name = "ICCIES-2025-DetectAI/vietnamese_news_human_ai"

In [129]:
dataset = load_dataset(dataset_name, split='train')
label_0_df = dataset.filter(lambda x: x["Label"] == 0).to_pandas()
sample_0_df = label_0_df.sample(n=12000, random_state=42)
label_1_df = dataset.filter(lambda x: x["Label"] == 1).to_pandas()
sample_1_df = label_1_df.sample(n=10000, random_state=42)
balanced_df = pd.concat([sample_0_df, sample_1_df])
final_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [131]:
os.makedirs("Data", exist_ok=True)
save_path = "Data/vietnamese_news_human_ai.csv"
final_df.to_csv(save_path, index=False, encoding='utf-8-sig')

## IV. Phân tích dữ liệu

### 1. Phân tích cơ bản (Có lọc mẫu)

In [155]:
class TextDataAnalyzer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = pd.read_csv(self.file_path)

    def add_text_features(self, text_col):
        texts = self.df[text_col].fillna("").astype(str)
        self.df['char_count'] = texts.apply(len)
        self.df['word_count'] = texts.apply(lambda x: len(x.split()))
        self.df['sentence_count'] = texts.apply(
            lambda x: max(1, len([s for s in re.split(r'[.!?]+', x) if s.strip()]))
        )


    def filter_by_word_count(self, text_col, min_words, max_words):
        if 'word_count' in self.df.columns:
            word_counts = self.df['word_count']
        else:
            word_counts = self.df[text_col].astype(str).apply(lambda x: len(x.split()))

        self.df = self.df[(word_counts >= min_words) & (word_counts <= max_words)]
        self.df = self.df.reset_index(drop=True)

        return len(self.df)

    def get_basic_stats(self, text_col, label_col="Label"):
        missing_counts = self.df.isnull().sum()

        label_counts = {}
        if label_col in self.df.columns:
            label_counts = self.df[label_col].value_counts().to_dict()

        return {
            "total_rows": len(self.df),
            "columns": list(self.df.columns),
            "duplicates": self.df.duplicated().sum(),
            "total_missing": missing_counts.sum(),
            "missing_details": missing_counts[missing_counts > 0].to_dict(),
            "empty_strings": (self.df[text_col].astype(str).str.strip() == "").sum(),
            "label_counts": label_counts
        }

    def get_text_stats(self, text_col):
        if self.df.empty:
            return {"min_len": 0, "max_len": 0, "mean_len": 0, "outliers_count": 0}

        if 'word_count' in self.df.columns:
            word_counts = self.df['word_count']
        else:
            word_counts = self.df[text_col].astype(str).apply(lambda x: len(x.split()))

        q1 = word_counts.quantile(0.25)
        q3 = word_counts.quantile(0.75)
        iqr = q3 - q1

        return {
            "min_len": word_counts.min(),
            "max_len": word_counts.max(),
            "mean_len": word_counts.mean(),
            "outliers_count": ((word_counts < q1 - 1.5 * iqr) | (word_counts > q3 + 1.5 * iqr)).sum()
        }

class ReportViewer:
    @staticmethod
    def display_report(file_path, text_col, basic_stats, text_stats):
        print(f"File: {file_path}")
        print("-" * 50)

        print("BÁO CÁO THỐNG KÊ CƠ BẢN")
        print(f"1. Tổng số mẫu (dòng): {basic_stats['total_rows']:,}")
        print(f"2. Các thuộc tính (cột): {basic_stats['columns']}")
        print(f"3. Dữ liệu trùng lặp: {basic_stats['duplicates']:,} dòng")
        print(f"4. Tổng giá trị Null/NaN: {basic_stats['total_missing']:,}")

        for col, count in basic_stats['missing_details'].items():
            print(f"   -> Cột '{col}' có {count:,} giá trị thiếu")

        print(f"5. Dữ liệu rỗng ở cột '{text_col}': {basic_stats['empty_strings']:,} dòng")

        if basic_stats.get('label_counts'):
            print("6. Phân bổ nhãn (Label):")
            for label, count in sorted(basic_stats['label_counts'].items()):
                print(f"   -> Label {label}: {count:,} mẫu")

        print("-" * 50)
        print("THỐNG KÊ CHIỀU DÀI VĂN BẢN")
        print(f"- Ngắn nhất: {text_stats['min_len']:,} từ")
        print(f"- Dài nhất: {text_stats['max_len']:,} từ")
        print(f"- Trung bình: {text_stats['mean_len']:,.2f} từ")
        print(f"- Số lượng Outliers (theo IQR): {text_stats['outliers_count']:,} mẫu")

In [156]:
analyzer = TextDataAnalyzer("Data/vietnamese_news_human_ai.csv")
data_column = "Text"
analyzer.add_text_features(text_col=data_column)

rows_after_filter = analyzer.filter_by_word_count(text_col=data_column, min_words=256, max_words=400)
basic_info = analyzer.get_basic_stats(text_col=data_column)
text_info = analyzer.get_text_stats(text_col=data_column)

ReportViewer.display_report(analyzer.file_path, data_column, basic_info, text_info)

Số mẫu còn lại sau khi lọc: 13,327 dòng

File: Data/vietnamese_news_human_ai.csv
--------------------------------------------------
BÁO CÁO THỐNG KÊ CƠ BẢN
1. Tổng số mẫu (dòng): 13,327
2. Các thuộc tính (cột): ['Text', 'Label', 'char_count', 'word_count', 'sentence_count']
3. Dữ liệu trùng lặp: 0 dòng
4. Tổng giá trị Null/NaN: 0
5. Dữ liệu rỗng ở cột 'Text': 0 dòng
6. Phân bổ nhãn (Label):
   -> Label 0: 6,569 mẫu
   -> Label 1: 6,758 mẫu
--------------------------------------------------
THỐNG KÊ CHIỀU DÀI VĂN BẢN
- Ngắn nhất: 256 từ
- Dài nhất: 400 từ
- Trung bình: 323.12 từ
- Số lượng Outliers (theo IQR): 0 mẫu


### 2. Phân tích nâng cao (đặc trưng)

In [154]:
DEFAULT_N_GRAM = 5
DEFAULT_TOP_K = 10

In [153]:
class TextFeatureExtractor:
    def __init__(self, df, text_col):
        self.df = df.dropna(subset=[text_col]).copy()
        self.text_col = text_col
        self.texts = self.df[text_col].astype(str)

    def compute_lexical_richness(self): # Type-Token Ratio - TTR
        def calc_ttr(text):
            tokens = text.lower().split()
            if not tokens:
                return 0.0
            types = set(tokens)
            return len(types) / len(tokens)

        self.df['ttr'] = self.texts.apply(calc_ttr)

        return self.df['ttr'].mean()

    def compute_structural_features(self):
        self.df['sentence_count'] = self.texts.apply(lambda x: max(1, len(re.split(r'[.!?]+', x)) - 1))
        self.df['word_count'] = self.texts.apply(lambda x: len(x.split()))
        self.df['number_count'] = self.texts.apply(lambda x: len(re.findall(r'\d+', x)))
        self.df['capitalized_words'] = self.texts.apply(lambda x: len(re.findall(r'\b[A-Z][a-z]*\b', x)))

        return {
            "avg_sentences": self.df['sentence_count'].mean(),
            "avg_words_per_sentence": (self.df['word_count'] / self.df['sentence_count']).mean(),
            "avg_numbers_per_text": self.df['number_count'].mean(),
            "avg_capitalized_words": self.df['capitalized_words'].mean()
        }

    def get_top_ngrams(self, n=DEFAULT_N_GRAM, top_k=DEFAULT_TOP_K): # N-gram phổ biến
        ngram_counts = Counter()
        for text in self.texts:
            tokens = re.findall(r'\b[a-z_àáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳỵỷỹ]+\b', text.lower())
            ngrams = zip(*[tokens[i:] for i in range(n)])

            for ngram in ngrams: # Nối cụm N-gram
                ngram_counts[" ".join(ngram)] += 1

        return ngram_counts.most_common(top_k)

class AdvancedReportViewer:
    @staticmethod
    def display_advanced_report(lexical_richness, structural_features, bigrams, trigrams):
        print("BÁO CÁO PHÂN TÍCH ĐẶC TRƯNG")
        print("-" * 65)

        print("1. ĐỘ PHONG PHÚ VÀ CẤU TRÚC NGÔN NGỮ:")
        print(f"   - Chỉ số đa dạng từ vựng (TTR trung bình): {lexical_richness:.4f}")
        print(f"   - Trung bình số câu / bài viết: {structural_features['avg_sentences']:.1f} câu")
        print(f"   - Trung bình số từ / câu: {structural_features['avg_words_per_sentence']:.1f} từ")
        print(f"   - Trung bình số lượng con số / bài: {structural_features['avg_numbers_per_text']:.1f} số")
        print(f"   - Trung bình số từ viết hoa / bài: {structural_features['avg_capitalized_words']:.1f} từ")

        print("\n2. PHÂN TÍCH XU HƯỚNG CỤM TỪ (N-GRAMS):")
        print("   -> Top 10 Cụm 2 từ xuất hiện nhiều nhất:")
        for phrase, count in bigrams:
            print(f"      + '{phrase}': {count:,} lần")

        print("   -> Top 10 Cụm 3 từ (Tri-grams) xuất hiện nhiều nhất:")
        for phrase, count in trigrams:
            print(f"      + '{phrase}': {count:,} lần")

In [157]:
extractor = TextFeatureExtractor(df=analyzer.df, text_col="Text")

ttr = extractor.compute_lexical_richness()
structural = extractor.compute_structural_features()
top_bigrams = extractor.get_top_ngrams(n=2, top_k=10)
top_trigrams = extractor.get_top_ngrams(n=3, top_k=10)

AdvancedReportViewer.display_advanced_report(
    lexical_richness=ttr,
    structural_features=structural,
    bigrams=top_bigrams,
    trigrams=top_trigrams,
)

BÁO CÁO PHÂN TÍCH ĐẶC TRƯNG
-----------------------------------------------------------------
1. ĐỘ PHONG PHÚ VÀ CẤU TRÚC NGÔN NGỮ:
   - Chỉ số đa dạng từ vựng (TTR trung bình): 0.6271
   - Trung bình số câu / bài viết: 11.2 câu
   - Trung bình số từ / câu: 32.4 từ
   - Trung bình số lượng con số / bài: 6.3 số
   - Trung bình số từ viết hoa / bài: 13.1 từ

2. PHÂN TÍCH XU HƯỚNG CỤM TỪ (N-GRAMS):
   -> Top 10 Cụm 2 từ xuất hiện nhiều nhất:
      + 'việt nam': 8,330 lần
      + 'có thể': 8,144 lần
      + 'không chỉ': 8,115 lần
      + 'là một': 7,770 lần
      + 'mà còn': 7,111 lần
      + 'phát triển': 6,003 lần
      + 'thực hiện': 5,688 lần
      + 'đặc biệt': 5,358 lần
      + 'quan trọng': 4,874 lần
      + 'chủ tịch': 4,788 lần
   -> Top 10 Cụm 3 từ (Tri-grams) xuất hiện nhiều nhất:
      + 'này không chỉ': 2,744 lần
      + 'mà còn là': 2,252 lần
      + 'một trong những': 2,056 lần
      + 'không chỉ là': 1,923 lần
      + 'đặc biệt là': 1,750 lần
      + 'mối quan hệ': 1,709 lầ

### 3. Phân tích độ nhiễu

In [164]:
class TextNoiseAnalyzer:
    def __init__(self, df, text_col):
        self.df = df.dropna(subset=[text_col]).copy()
        self.text_col = text_col
        self.texts = self.df[text_col].astype(str)

    def analyze_formatting_noise(self):
        self.df['email_count'] = self.texts.apply(lambda x: len(re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', x)))
        return {"avg_emails": self.df['email_count'].mean()}

    def analyze_lexical_noise(self):
        self.df['long_words_count'] = self.texts.apply(lambda x: len(re.findall(r'\b\w{15,}\b', x)))
        vietnamese_chars_pattern = r'[^a-zA-Z0-9\s_àáãạảăắằẳẵặâấầẩẫậèéẹẻẽêềếểễệđìíĩỉịòóõọỏôốồổỗộơớờởỡợùúũụủưứừửữựỳỵỷỹÀÁÃẠẢĂẮẰẲẴẶÂẤẦẨẪẬÈÉẸẺẼÊỀẾỂỄỆĐÌÍĨỈỊÒÓÕỌỎÔỐỒỔỖỘƠỚỜỞỠỢÙÚŨỤỦƯỨỪỬỮỰỲỴỶỸ]'
        special_chars = self.texts.apply(lambda x: len(re.findall(vietnamese_chars_pattern, x)))
        text_lengths = self.texts.apply(len).replace(0, 1)
        self.df['special_char_ratio'] = special_chars / text_lengths

        return {
            "avg_long_words": self.df['long_words_count'].mean(),
            "special_char_ratio": self.df['special_char_ratio'].mean()
        }

    def analyze_syntactic_noise(self):
        self.df['caps_ratio'] = self.texts.apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0)
        self.df['missing_punctuation'] = self.texts.apply(lambda x: 1 if len(re.findall(r'[.!?]', x)) == 0 else 0)

        return {
            "avg_caps_ratio": self.df['caps_ratio'].mean(),
            "missing_punctuation_ratio": self.df['missing_punctuation'].mean()
        }

class NoiseReportViewer:
    @staticmethod
    def display_noise_report(formatting, lexical, syntactic):
        print("BÁO CÁO PHÂN TÍCH ĐỘ NHIỄU DỮ LIỆU")
        print("-" * 65)

        print("1. NHIỄU ĐỊNH DẠNG:")
        print(f"   - Trung bình Email / bài: {formatting['avg_emails']:.2f} email")

        print("\n2. NHIỄU TỪ VỰNG & KÝ TỰ:")
        print(f"   - Trung bình từ dính chùm (> 15 ký tự) / bài: {lexical['avg_long_words']:.2f} từ")
        print(f"   - Tỷ lệ ký tự lạ / đặc biệt: {lexical['special_char_ratio']:.4%}")

        print("\n3. NHIỄU CÚ PHÁP:")
        print(f"   - Tỷ lệ chữ IN HOA toàn văn bản: {syntactic['avg_caps_ratio']:.4%}")
        print(f"   - Tỷ lệ bài viết KHÔNG CÓ dấu chấm câu: {syntactic['missing_punctuation_ratio']:.4%}")

In [168]:
noise_analyzer = TextNoiseAnalyzer(df=analyzer.df, text_col="Text")

formatting_noise = noise_analyzer.analyze_formatting_noise()
lexical_noise = noise_analyzer.analyze_lexical_noise()
syntactic_noise = noise_analyzer.analyze_syntactic_noise()

NoiseReportViewer.display_noise_report(formatting_noise, lexical_noise, syntactic_noise)
analyzer.df = noise_analyzer.df

BÁO CÁO PHÂN TÍCH ĐỘ NHIỄU DỮ LIỆU
-----------------------------------------------------------------
1. NHIỄU ĐỊNH DẠNG:
   - Trung bình Email / bài: 0.01 email

2. NHIỄU TỪ VỰNG & KÝ TỰ:
   - Trung bình từ dính chùm (> 15 ký tự) / bài: 0.02 từ
   - Tỷ lệ ký tự lạ / đặc biệt: 2.6476%

3. NHIỄU CÚ PHÁP:
   - Tỷ lệ chữ IN HOA toàn văn bản: 2.9913%
   - Tỷ lệ bài viết KHÔNG CÓ dấu chấm câu: 0.0525%


### 4. Xét đặc trưng quan trọng

In [160]:
class VietnameseDataProcessor:
    def __init__(self, custom_stopwords=None):
        self.stopwords = set()
        self.load_stopwords(custom_stopwords)


    def load_stopwords(self, file_path):
        if os.path.exists(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    self.stopwords = set(line.strip().replace(' ', '_') for line in f if line.strip())
            except Exception as e:
                self.stopwords = set(["và", "thì", "là", "mà", "của", "đã", "đang", "sẽ", "để", "các", "những", "một"])
        else:
            self.stopwords = set(["và", "thì", "là", "mà", "của", "đã", "đang", "sẽ", "để", "các", "những", "một"])

    def segment_words(self, text):
        if not text or not isinstance(text, str):
            return ""

        return word_tokenize(text, format="text")

    def calculate_stopwords_ratio(self, text):
        tokens = text.split()
        if not tokens:
            return 0.0, []

        stopword_tokens = [word for word in tokens if word.lower() in self.stopwords]
        ratio = (len(stopword_tokens) / len(tokens)) * 100

        return ratio, stopword_tokens

    def detect_language(self, text):
        try:
            main_lang = detect(text)
            return main_lang
        except:
            return "unknown"

    def analyze_document(self, text):
        segmented_text = self.segment_words(text)

        sw_ratio, sw_found = self.calculate_stopwords_ratio(segmented_text)

        lang = self.detect_language(text)

        return {
            "original_text": text,
            "segmented_text": segmented_text,
            "stopwords_ratio_percent": round(sw_ratio, 2),
            "stopwords_found": sw_found,
            "main_language": lang,
            "is_valid_vietnamese": lang == "vi"
        }

In [161]:
class DatasetNLPAnalyzer:
    def __init__(self, dataframe, text_col, nlp_processor):
        self.df = dataframe
        self.text_col = text_col
        self.processor = nlp_processor
        self.stats = {}

    def process_data(self):
        self.df[self.text_col] = self.df[self.text_col].fillna("")

        self.df['nlp_results'] = self.df[self.text_col].apply(
            lambda x: self.processor.analyze_document(str(x))
        )

        self.df['segmented_text'] = self.df['nlp_results'].apply(lambda x: x['segmented_text'])
        self.df['stopwords_ratio'] = self.df['nlp_results'].apply(lambda x: x['stopwords_ratio_percent'])
        self.df['language'] = self.df['nlp_results'].apply(lambda x: x['main_language'])
        self.df['is_valid_vi'] = self.df['nlp_results'].apply(lambda x: x['is_valid_vietnamese'])
        self.df['stopwords_found'] = self.df['nlp_results'].apply(lambda x: x['stopwords_found'])

        self.df = self.df.drop(columns=['nlp_results'])

    def calculate_statistics(self):
        self.stats['total_docs'] = len(self.df)
        self.stats['lang_stats'] = self.df['language'].value_counts(normalize=True) * 100
        self.stats['valid_vi_percent'] = self.df['is_valid_vi'].mean() * 100
        self.stats['avg_stopwords_ratio'] = self.df['stopwords_ratio'].mean()

        all_stopwords = []
        for sw_list in self.df['stopwords_found']:
            all_stopwords.extend(sw_list)
        self.stats['top_10_stopwords'] = Counter(all_stopwords).most_common(10)

    def print_report(self):
        print("\n" + "-"*50)
        print(" BÁO CÁO THỐNG KÊ XỬ LÝ NGÔN NGỮ TỰ NHIÊN (NLP)")
        print("-"*50)
        print(f"Tổng số văn bản đã xử lý: {self.stats['total_docs']:,} dòng")

        print("\n1. THỐNG KÊ TÌNH TRẠNG NGÔN NGỮ:")
        print(f" - Tỷ lệ văn bản Tiếng Việt (Hợp lệ): {self.stats['valid_vi_percent']:.2f}%")
        print(f" - Tỷ lệ ngôn ngữ ngoại lai / nhiễu: {(100 - self.stats['valid_vi_percent']):.2f}%")
        print(" - Chi tiết phân bổ các mã ngôn ngữ:")
        for lang, pct in self.stats['lang_stats'].items():
            print(f"   + Mã '{lang}': {pct:.2f}%")

        print("\n2. THỐNG KÊ TỶ LỆ STOPWORDS (HƯ TỪ):")
        print(f" - Tỷ lệ Stopwords trung bình trên toàn tập: {self.stats['avg_stopwords_ratio']:.2f}%")
        print(" - Top 10 Stopwords làm nhiễu dữ liệu nhiều nhất:")
        for word, count in self.stats['top_10_stopwords']:
            print(f"   + Từ '{word}': {count:,} lần xuất hiện")

    def run_pipeline(self):
        self.process_data()
        self.calculate_statistics()
        self.print_report()
        return self.df

In [162]:
processor = VietnameseDataProcessor(custom_stopwords="Data/vietnamese-stopwords.txt")

dataset_analyzer = DatasetNLPAnalyzer(
    dataframe=analyzer.df,
    text_col='Text',
    nlp_processor=processor
)

analyzer.df = dataset_analyzer.run_pipeline()


--------------------------------------------------
 BÁO CÁO THỐNG KÊ XỬ LÝ NGÔN NGỮ TỰ NHIÊN (NLP)
--------------------------------------------------
Tổng số văn bản đã xử lý: 13,327 dòng

1. THỐNG KÊ TÌNH TRẠNG NGÔN NGỮ:
 - Tỷ lệ văn bản Tiếng Việt (Hợp lệ): 99.99%
 - Tỷ lệ ngôn ngữ ngoại lai / nhiễu: 0.01%
 - Chi tiết phân bổ các mã ngôn ngữ:
   + Mã 'vi': 99.99%
   + Mã 'en': 0.01%

2. THỐNG KÊ TỶ LỆ STOPWORDS (HƯ TỪ):
 - Tỷ lệ Stopwords trung bình trên toàn tập: 42.88%
 - Top 10 Stopwords làm nhiễu dữ liệu nhiều nhất:
   + Từ 'và': 71,360 lần xuất hiện
   + Từ 'của': 56,939 lần xuất hiện
   + Từ 'trong': 38,231 lần xuất hiện
   + Từ 'là': 37,406 lần xuất hiện
   + Từ 'những': 37,039 lần xuất hiện
   + Từ 'một': 33,227 lần xuất hiện
   + Từ 'đã': 32,428 lần xuất hiện
   + Từ 'các': 32,142 lần xuất hiện
   + Từ 'cho': 30,182 lần xuất hiện
   + Từ 'với': 28,549 lần xuất hiện


In [170]:
clean_df = analyzer.df[analyzer.df['is_valid_vi'] == True]
pd.set_option('display.max_colwidth', None)
clean_df.head(2)


,Text,Label,char_count,word_count,sentence_count,segmented_text,stopwords_ratio,language,is_valid_vi,stopwords_found,email_count,long_words_count,special_char_ratio,caps_ratio,missing_punctuation
0,"Amandine, một cô gái trẻ đầy nhiệt huyết, đã từng theo học ngành thiết kế thời trang trước khi quyết định chuyển đến Melbourne, Australia trong một năm để theo đuổi đam mê ẩm thực. Sau khi trở về quê hương Pháp, cô bắt đầu làm việc như một nhà thiết kế thực phẩm. Thời gian rảnh rỗi, Amandine thường ngồi cùng bạn bè tại những quán café ngoài trời, tận hưởng không khí và quan sát những người đi qua. Tuy nhiên, cô cảm thấy cuộc sống của mình có phần đơn điệu và thiếu màu sắc. Người Paris thường nói về cuộc sống đều đặn của họ bằng câu nói “metro, boulot, dodo”, nghĩa là “di chuyển, làm việc, ngủ”. Amandine nhận ra rằng cô cần một sự thay đổi, một nơi chốn mới để khám phá những thử thách khác.\n\nVì vậy, cô quyết định trở về Việt Nam, chọn sống tại quận 2, TP. HCM và bắt đầu học tiếng Việt từ đầu. Một trong những thay đổi lớn nhất trong cuộc sống của Amandine khi chuyển đến đây là việc cô đã chuyển sang chế độ ăn chay. Sau khi nhận ra rằng việc tiêu thụ quá nhiều thịt không tốt cho sức khỏe, cô đã quyết định từ bỏ nó. Cơ thể và tâm hồn cô dần thích nghi với lối sống mới này. Mỗi sáng, Amandine thức dậy sớm, bắt đầu ngày mới bằng một cốc nước lớn, tiếp theo là một tách cà phê đen nóng không đường, và dành một giờ để tập yoga trước khi bắt tay vào công việc. Cuộc sống của cô giờ đây trở nên ngăn nắp và lành mạnh hơn rất nhiều so với thời gian ở Paris. Khi được hỏi về trang phục khiến cô tự tin nhất, Amandine cho biết nó phụ thuộc vào tâm trạng của cô trong từng khoảnh khắc.",1,1490,318,14,"Amandine , một cô gái trẻ đầy nhiệt_huyết , đã từng theo học ngành thiết_kế thời_trang trước khi quyết_định chuyển đến Melbourne , Australia trong một năm để theo_đuổi đam_mê ẩm_thực . Sau khi trở về quê_hương Pháp , cô bắt_đầu làm_việc như một nhà_thiết_kế thực_phẩm . Thời_gian rảnh_rỗi , Amandine_thường ngồi cùng bạn_bè tại những quán café ngoài_trời , tận_hưởng không_khí và quan_sát những người đi qua . Tuy_nhiên , cô cảm_thấy cuộc_sống của mình có phần đơn_điệu và thiếu màu_sắc . Người Paris_thường nói về cuộc_sống đều_đặn của họ bằng câu nói “ metro , boulot , dodo ” , nghĩa_là “ di_chuyển , làm_việc , ngủ ” . Amandine nhận ra rằng cô cần một sự thay_đổi , một nơi chốn mới để khám_phá những thử_thách khác . Vì_vậy , cô quyết_định trở về Việt_Nam , chọn sống tại quận 2 , TP. HCM và bắt_đầu học tiếng Việt từ đầu . Một trong những thay_đổi lớn nhất trong cuộc_sống của Amandine khi chuyển đến đây là việc cô đã chuyển sang chế_độ ăn_chay . Sau khi nhận ra rằng việc tiêu_thụ quá nhiều thịt không tốt cho sức_khỏe , cô đã quyết_định từ_bỏ nó . Cơ_thể và tâm_hồn cô dần thích_nghi với lối sống mới này . Mỗi sáng , Amandine_thức dậy sớm , bắt_đầu ngày mới bằng một cốc nước_lớn , tiếp_theo là một tách cà_phê đen nóng không đường , và dành một giờ để tập_yoga trước khi bắt_tay vào công_việc . Cuộc_sống của cô giờ_đây trở_nên ngăn_nắp và lành_mạnh hơn rất nhiều so với thời_gian ở Paris . Khi được hỏi về trang_phục khiến cô tự_tin nhất , Amandine cho biết nó phụ_thuộc vào tâm_trạng của cô trong từng khoảnh_khắc .",52.78,vi,True,"[một, cô, đầy, đã, từng, theo, trước, khi, chuyển, đến, trong, một, năm, để, Sau, khi, về, cô, bắt_đầu, như, một, Thời_gian, ngồi, cùng, tại, những, và, những, người, qua, Tuy_nhiên, cô, cảm_thấy, của, mình, có, phần, và, thiếu, Người, nói, về, của, họ, bằng, nói, nhận, ra, rằng, cô, cần, một, sự, thay_đổi, một, nơi, mới, để, những, khác, Vì_vậy, cô, về, chọn, tại, quận, và, bắt_đầu, từ, Một, trong, những, thay_đổi, lớn, nhất, trong, của, khi, chuyển, đến, đây, là, việc, cô, đã, chuyển, sang, Sau, khi, nhận, ra, rằng, việc, quá, nhiều, không, tốt, cho, cô, đã, ...]",0,0,0.026846,0.020134,0
1,"thể thuê tài xế nhưng thích tự lái xe đưa con đi học mỗi sáng, sau đó lang bang lượn phố vài vòng shopping trước khi trở về với đống phim bộ ở nhà.

In [114]:
pd.reset_option('display.max_colwidth')

In [171]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13326 entries, 0 to 13326
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Text                 13326 non-null  object 
 1   Label                13326 non-null  int64  
 2   char_count           13326 non-null  int64  
 3   word_count           13326 non-null  int64  
 4   sentence_count       13326 non-null  int64  
 5   segmented_text       13326 non-null  object 
 6   stopwords_ratio      13326 non-null  float64
 7   language             13326 non-null  object 
 8   is_valid_vi          13326 non-null  bool   
 9   stopwords_found      13326 non-null  object 
 10  email_count          13326 non-null  int64  
 11  long_words_count     13326 non-null  int64  
 12  special_char_ratio   13326 non-null  float64
 13  caps_ratio           13326 non-null  float64
 14  missing_punctuation  13326 non-null  int64  
dtypes: bool(1), float64(3), int64(7), object(

In [172]:
save_path = "Data/vietnamese_news_human_ai_eda.csv"
clean_df.to_csv(save_path, index=False, encoding='utf-8-sig')